# Adapter Subgroup Analysis — Paper Figures

Reproduces Table 1, Figure 2, Table 2 and Figure 3 from ["Subgroup performance
analysis of adaptation strategies for chest X-ray foundation models"](https://arxiv.org/abs/2608.19078)
(Gupta et al., MICCAI 2026 FAIMI Workshop).

Run the pipeline in `scripts/` first (see the README) to produce the checkpoints
and evaluation CSVs this notebook reads — no training or backbone inference
happens here except for scoring the two cached-embedding checkpoints, which is
cheap.


In [ ]:
import sys
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, "..")  # repo root — for `paths` and `evaluation`/`training`
import paths
from evaluation.common import (
    ATTRIBUTE_ROWS,
    DEMO_ORDER,
    PATHOLOGY_COL_MAP,
    PATHOLOGY_TASKS,
    PATH_SHORT,
    attribute_rows,
    build_joint_sets,
    load_test_df,
    pathology_rows,
    score_cached_checkpoint,
)

sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.family": "sans-serif"})

CHECKPOINT_DIR = Path(paths.CHECKPOINT_DIR)
OUTPUT_DIR = Path(paths.OUTPUT_DIR)
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATASET_FILTER = 1     # MIMIC-CXR
N_FACTOR = 3            # resampled-cohort size multiplier
RESAMPLE_SEED = 42      # point-estimate seed, matches evaluation/eval_bootstrap_ci.py

ATTR_LABELS = {
    "race_Asian": "Race: Asian", "race_Black": "Race: Black", "race_White": "Race: White",
    "sex": "Sex: Male", "view_AP": "View: AP", "view_PA": "View: PA", "view_Lateral": "View: Lateral",
}
ROW_ORDER = PATHOLOGY_TASKS + ATTRIBUTE_ROWS
ROW_LABELS = {**PATHOLOGY_COL_MAP, **ATTR_LABELS}

METHOD_ORDER = ["No Adapter", "MLP", "Attention Pooling"]
METHOD_COLORS = {"No Adapter": "#0070c0", "MLP": "#ff9300", "Attention Pooling": "#4ea72e"}

CONFIG_LABELS = {
    "attn_early_layers": "Early Layers", "attn_late_layers": "Late Layers",
    "attn_split_layers": "Split Layers", "attn_even_layers": "Even Layers",
}
CONFIG_ORDER = ["Early Layers", "Late Layers", "Split Layers", "Even Layers"]
CONFIG_COLORS = {"Early Layers": "#0070c0", "Late Layers": "#ff9300", "Split Layers": "#4ea72e", "Even Layers": "#ff7e79"}


def styled_ci_table(ci_df, method_order):
    """Pivot a long-format [method, task, point_estimate, ci_lower, ci_upper] CI
    table into the paper's Table 1/2 layout: one row per task, one column per
    method, cell = 'point_estimate [ci_lower, ci_upper]', bolding the row max."""
    pe = ci_df.pivot(index="task", columns="method", values="point_estimate").reindex(index=ROW_ORDER, columns=method_order)
    lo = ci_df.pivot(index="task", columns="method", values="ci_lower").reindex(index=ROW_ORDER, columns=method_order)
    hi = ci_df.pivot(index="task", columns="method", values="ci_upper").reindex(index=ROW_ORDER, columns=method_order)

    text = pd.DataFrame(index=pe.index, columns=pe.columns, dtype=object)
    for r in pe.index:
        for c in pe.columns:
            v, l, h = pe.loc[r, c], lo.loc[r, c], hi.loc[r, c]
            text.loc[r, c] = f"{v:.3f} [{l:.3f}, {h:.3f}]" if pd.notna(v) else "\u2014"
    text.index = [ROW_LABELS.get(t, t) for t in text.index]

    is_max = pe.eq(pe.max(axis=1), axis=0).values

    def _bold(_):
        return pd.DataFrame(np.where(is_max, "font-weight: bold", ""), index=text.index, columns=text.columns)

    return text.style.apply(_bold, axis=None)


def grouped_bar_auroc(df, x_order, x_labels, group_order, colors, title, save_name, y_min, y_max):
    """One grouped bar chart: x = task, one bar per `method` value."""
    x_indices = np.arange(len(x_order))
    bar_width = 0.8 / len(group_order)
    offsets = np.linspace(-0.4 + bar_width / 2, 0.4 - bar_width / 2, len(group_order))

    fig, ax = plt.subplots(figsize=(max(10, len(x_order) * 1.3), 6))
    for group, offset in zip(group_order, offsets):
        g_df = df[df["method"] == group].set_index("task").reindex(x_order)
        ax.bar(
            x_indices + offset, g_df["auroc"].values, width=bar_width,
            label=group, color=colors[group], edgecolor="#2A2A2A", linewidth=1.0, zorder=3,
        )

    ax.set_xticks(x_indices)
    ax.set_xticklabels(x_labels, rotation=30, ha="right", fontsize=14)
    ax.set_ylabel("AUROC", fontsize=16, fontweight="bold")
    ax.set_ylim(y_min, y_max)
    ax.set_title(title, fontsize=18, fontweight="bold", pad=15)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=11, loc="lower right")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{save_name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.savefig(FIGURES_DIR / f"{save_name}.pdf", bbox_inches="tight", facecolor="white")
    plt.show()


def disparity_heatmap(df, group_order, title, save_name):
    """Subgroup AUROC minus the OVERALL AUROC (the subgroup=="all" row: one AUROC
    computed on the full resampled cohort, pooling every subgroup together) for
    each (method, task) — in percentage points."""
    overall = df[df["subgroup"] == "all"][["task", "method", "auroc"]].rename(columns={"auroc": "overall_auroc"})
    d = df[df["subgroup"].isin(DEMO_ORDER)].copy()
    d = d.merge(overall, on=["task", "method"], how="left")
    missing = d[d["overall_auroc"].isna()][["task", "method"]].drop_duplicates()
    if len(missing):
        print(f"no overall ('all') AUROC found for {len(missing)} (task, method) pairs:")
        print(missing.to_string(index=False))
    d["Disparity"] = (d["auroc"] - d["overall_auroc"]) * 100
    d["Class"] = d["task"].map(PATH_SHORT)

    n = len(group_order)
    fig, axes = plt.subplots(1, n + 1, figsize=(7 * n, 8), gridspec_kw={"width_ratios": [1] * n + [0.05]})
    plot_axes, cbar_ax = axes[:-1], axes[-1]
    norm = mcolors.TwoSlopeNorm(vmin=-8, vcenter=0, vmax=8)

    for idx, group in enumerate(group_order):
        ax = plot_axes[idx]
        g_df = d[d["method"] == group]
        pivot = g_df.pivot(index="Class", columns="subgroup", values="Disparity")
        pivot = pivot.reindex(index=list(PATH_SHORT.values()), columns=DEMO_ORDER)
        sns.heatmap(
            pivot, cmap="RdBu_r", norm=norm, annot=True, fmt=".1f", linewidths=0.7,
            annot_kws={"size": 12, "weight": "bold"}, square=True,
            cbar=(idx == n - 1), cbar_ax=cbar_ax if idx == n - 1 else None, ax=ax,
        )
        ax.set_xlabel("Attribute" if idx == n // 2 else "", fontsize=16, fontweight="bold")
        ax.set_ylabel("Class" if idx == 0 else "", fontsize=16, fontweight="bold")
        if idx > 0:
            ax.set_yticklabels([])
        ax.tick_params(axis="both", labelsize=12)
        ax.tick_params(axis="y", rotation=30)
        ax.set_title(group, fontsize=17, fontweight="bold", pad=12)

    cbar_ax.set_ylabel("Difference from overall AUROC (%)", fontsize=14, fontweight="bold", labelpad=12)
    cbar_ax.tick_params(labelsize=12)
    fig.suptitle(title, fontsize=20, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{save_name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.savefig(FIGURES_DIR / f"{save_name}.pdf", bbox_inches="tight", facecolor="white")
    plt.show()


## Dataset overview

In [ ]:
# Pathology prevalence on the MIMIC-CXR test set, under the NaN-as-negative
# label convention used throughout training and evaluation: positive iff a
# pathology is explicitly mentioned (column value 2); everything else
# (unmentioned, explicit negative, uncertain) counts as negative.
test_df_full, _ = load_test_df(str(paths.METADATA_CSV), DATASET_FILTER)
n_total = len(test_df_full)

prev_df = pd.DataFrame([
    dict(task=task, label=PATHOLOGY_COL_MAP[task],
         n_pos=int((test_df_full[col] == 2).sum()))
    for task, col in PATHOLOGY_COL_MAP.items()
])
prev_df["prevalence"] = prev_df["n_pos"] / n_total
prev_df = prev_df.sort_values("prevalence")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(prev_df["label"], prev_df["prevalence"] * 100,
                color=plt.get_cmap("tab10").colors[:len(prev_df)],
                edgecolor="#2A2A2A", linewidth=1.0, zorder=3)
for bar, n_pos in zip(bars, prev_df["n_pos"]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.1f}%", va="center", fontsize=11)
ax.set_xlabel("Prevalence (%)", fontsize=13, fontweight="bold")
ax.set_title(f"Pathology prevalence on the MIMIC-CXR test set (n={n_total:,})",
             fontsize=14, fontweight="bold")
ax.set_xlim(0, prev_df["prevalence"].max() * 100 * 1.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dataset_pathology_prevalence.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()


## Table 1 & Figure 2 — No Adapter vs MLP vs Attention Pooling

In [ ]:
# Score No Adapter + MLP directly from cached CLS embeddings — cheap enough to do
# inline (no backbone forward pass needed, both operate on a fixed 768-dim
# embedding), so no separate CLI script/CSV round-trip is needed for these two.
test_df, n_full_test = load_test_df(str(paths.METADATA_CSV), DATASET_FILTER)
dat_path = Path(paths.EMBEDDING_CACHE_DIR) / "raddino_emb_float32_test.dat"
dat = np.memmap(dat_path, dtype="float32", mode="r", shape=(n_full_test, 768))
X_test = np.array(dat[test_df["_embed_idx"].values], dtype=np.float32)

joint_sets = build_joint_sets(test_df, N_FACTOR, RESAMPLE_SEED)

cached_path_rows, cached_attr_rows = [], []
for ckpt_name in ["no_adapter", "mlp_adapter"]:
    method, pathology_scores, attribute_scores = score_cached_checkpoint(ckpt_name, CHECKPOINT_DIR, X_test)
    cached_path_rows += pathology_rows(method, ckpt_name, pathology_scores, joint_sets)
    cached_attr_rows += attribute_rows(method, ckpt_name, attribute_scores, test_df)

df_cached_path = pd.DataFrame(cached_path_rows)
df_cached_attr = pd.DataFrame(cached_attr_rows)
print(f"Scored No Adapter + MLP: {len(df_cached_path)} pathology rows, {len(df_cached_attr)} attribute rows")


In [ ]:
# Attention pooling needs a live backbone pass, so its point estimates come from
# scripts/eval_attn_multilayer.sh and scripts/eval_attn_attributes.sh instead —
# load their output CSVs here.
EVAL_DIR = OUTPUT_DIR / "eval"

df_attn_path = pd.read_csv(EVAL_DIR / "attn_multilayer_eval.csv")
df_attn_path["method"] = df_attn_path["checkpoint"].map(CONFIG_LABELS)  # per-layer-config label

df_attn_attr = pd.read_csv(EVAL_DIR / "attn_probe_attributes_eval.csv")
df_attn_attr["method"] = df_attn_attr["base_checkpoint"].map(CONFIG_LABELS)

# 3-way comparison uses only the "Even Layers" checkpoint, relabelled "Attention Pooling".
attn_path_even = df_attn_path[df_attn_path["checkpoint"] == "attn_even_layers"].copy()
attn_path_even["method"] = "Attention Pooling"
attn_attr_even = df_attn_attr[df_attn_attr["base_checkpoint"] == "attn_even_layers"].copy()
attn_attr_even["method"] = "Attention Pooling"

df_eval = pd.concat([df_cached_path, attn_path_even[df_cached_path.columns]], ignore_index=True)
df_attr_eval = pd.concat([df_cached_attr, attn_attr_even[df_cached_attr.columns]], ignore_index=True)

eval_methods = sorted(df_eval["method"].unique())
attr_methods = sorted(df_attr_eval["method"].unique())
print(f"df_eval methods: {eval_methods}")
print(f"df_attr_eval methods: {attr_methods}")


In [ ]:
# Table 1 — overall AUROC with 95% CI (produced by scripts/eval_bootstrap_ci.sh)
ci_table_3way = pd.read_csv(OUTPUT_DIR / "eval/bootstrap/1_global_table_3way_summary.csv")
styled_ci_table(ci_table_3way, METHOD_ORDER)


In [ ]:
# Table 1 companion bar charts — point estimates only (no CI), one per category.
path_overall = df_eval[(df_eval["subgroup_type"] == "overall") & (df_eval["subgroup"] == "all")]
grouped_bar_auroc(
    path_overall, x_order=PATHOLOGY_TASKS, x_labels=[PATH_SHORT[t] for t in PATHOLOGY_TASKS],
    group_order=METHOD_ORDER, colors=METHOD_COLORS,
    title="Pathology Classification", save_name="method_comparison_pathology_auroc",
    y_min=0.7, y_max=1.01,
)
grouped_bar_auroc(
    df_attr_eval, x_order=ATTRIBUTE_ROWS, x_labels=[ATTR_LABELS[t].split(": ")[-1] for t in ATTRIBUTE_ROWS],
    group_order=METHOD_ORDER, colors=METHOD_COLORS,
    title="Attribute Classification", save_name="method_comparison_attribute_auroc",
    y_min=0.7, y_max=1.01,
)


In [ ]:
# Figure 2 — subgroup performance disparity heatmap
disparity_heatmap(
    df_eval, group_order=METHOD_ORDER,
    title="Subgroup performance disparity — No Adapter vs MLP vs Attention Pooling",
    save_name="subgroup_heatmap_3way",
)


## Table 2 & Figure 3 — Attention-Pooling layer configurations

In [ ]:
# Table 2 — overall AUROC with 95% CI, across the 4 layer configs
ci_table_4config = pd.read_csv(OUTPUT_DIR / "eval/bootstrap/3_global_table_4config_summary.csv")
styled_ci_table(ci_table_4config, CONFIG_ORDER)


In [ ]:
# Table 2 companion bar charts
config_path_overall = df_attn_path[(df_attn_path["subgroup_type"] == "overall") & (df_attn_path["subgroup"] == "all")]
grouped_bar_auroc(
    config_path_overall, x_order=PATHOLOGY_TASKS, x_labels=[PATH_SHORT[t] for t in PATHOLOGY_TASKS],
    group_order=CONFIG_ORDER, colors=CONFIG_COLORS,
    title="Pathology Classification", save_name="layer_config_comparison_pathology_auroc",
    y_min=0.75, y_max=1.01,
)
grouped_bar_auroc(
    df_attn_attr, x_order=ATTRIBUTE_ROWS, x_labels=[ATTR_LABELS[t].split(": ")[-1] for t in ATTRIBUTE_ROWS],
    group_order=CONFIG_ORDER, colors=CONFIG_COLORS,
    title="Attribute Classification", save_name="layer_config_comparison_attribute_auroc",
    y_min=0.75, y_max=1.01,
)


In [ ]:
# Figure 3 — subgroup performance disparity heatmap across layer configs
disparity_heatmap(
    df_attn_path, group_order=CONFIG_ORDER,
    title="Subgroup performance disparity — attention-pooling layer configurations",
    save_name="subgroup_heatmap_layer_configs",
)
